# Sharpe Smoothing Explorer

Notebook version of the interactive explorer. Choose a true Sharpe, draw a noisy ten-year realization, and compare raw $\hat S$ against five fixed-halflife EMAs and three adaptive ones. Everything is shown both as $\hat S$ and as the allocation score $T\hat S$. Accuracy and turnover are measured over an ensemble of independent paths.

**Model.** Daily excess returns are i.i.d. with annualized Sharpe $S$. The estimate uses an expanding window from inception, $N_t = t+1$ days and $T_t = N_t/252$ years:

$$
\hat S_t \;=\; S + \sqrt{252}\,\frac{1}{N_t}\sum_{i\le t}\varepsilon_i,
\qquad \varepsilon_i \sim \mathcal N(0,1),
\qquad \operatorname{Var}(\hat S_t) = \frac{1}{T_t}.
$$

**Filters.** Every filter runs on daily data. An EMA with halflife $h$ has weight $\alpha = 1-2^{-1/h}$. For the first observations it acts as a running mean:

$$
y_t = y_{t-1} + \alpha_t\,(\hat S_t - y_{t-1}),
\qquad \alpha_t = \max\!\Big(\alpha,\ \tfrac{1}{n_t}\Big).
$$

The adaptive filters set the halflife to a fixed fraction of the window, $h_t = \max(3,\ N_t/d)$ with $d \in \{15, 30, 60\}$.

**Metrics.** These are computed on an observation grid $t_0 < t_1 < \dots$ spaced $k$ days apart, pooled over $P$ paths:

$$
\mathrm{RMSE}_{\hat S} = \sqrt{\tfrac{1}{PG}\textstyle\sum_{p,q}\big(y^{(p)}_{t_q}-S\big)^2},
\qquad
\mathrm{RMSE}_{T\hat S} = \sqrt{\tfrac{1}{PG}\textstyle\sum_{p,q}T_{t_q}^2\big(y^{(p)}_{t_q}-S\big)^2},
$$

$$
\mathrm{TO} = \frac{252}{k}\cdot\operatorname{mean}_{p,q}\Big|T_{t_q}y_{t_q}-T_{t_{q-1}}y_{t_{q-1}}\Big|,
\qquad
T_{\text{eff}} = \frac{1}{\operatorname{Var}(y-S)}.
$$

Turnover here is the annualized total variation of one strategy's score. It is a proxy: actual trading comes from the normalized weights, where moves shared across strategies partly cancel.

In [1]:
# If needed:  %pip install numpy pandas plotly ipywidgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as W
    from IPython.display import display, clear_output
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not found - the static cells still work; install it for the interactive panel.")

DPY, YEARS = 252, 10
N = DPY * YEARS
T = np.arange(1, N + 1) / DPY          # window length in years, expanding from day 1

## Series and colours

Fixed halflives use a blue ramp that runs light to dark as the halflife lengthens. Adaptive rules use an orange ramp in the same order, with distinct dash patterns. Raw is grey.

In [2]:
SERIES = [
    # id,     label,         kind,   param, colour,    dash
    ("raw",   "raw Ŝ",       "raw",  None,  "#8d928f", "solid"),
    ("h5",    "EMA 1w",      "ema",  5,     "#86b6ef", "solid"),
    ("h21",   "EMA 1m",      "ema",  21,    "#5598e7", "solid"),
    ("h63",   "EMA 3m",      "ema",  63,    "#2a78d6", "solid"),
    ("h126",  "EMA 6m",      "ema",  126,   "#184f95", "solid"),
    ("h252",  "EMA 1y",      "ema",  252,   "#0d366b", "solid"),
    ("ada60", "EMA h=N/60",  "ada",  60,    "#ee8a4f", "dot"),
    ("ada30", "EMA h=N/30",  "ada",  30,    "#cf5a22", "dash"),
    ("ada15", "EMA h=N/15",  "ada",  15,    "#8f3712", "longdash"),
]
IDS    = [s[0] for s in SERIES]
LABEL  = {s[0]: s[1] for s in SERIES}
COLOR  = {s[0]: s[4] for s in SERIES}
DASH   = {s[0]: s[5] for s in SERIES}
FREQS  = {"1d": 1, "1w": 5, "1m": 21, "3m": 63}

## Simulation and filters

Every path runs through all nine filters in a single vectorized pass over time. The array has shape (paths, filters), so 400 paths take a fraction of a second.

In [3]:
def raw_paths(S, n_paths, seed):
    rng = np.random.default_rng(seed)
    eps = rng.standard_normal((n_paths, N))
    return S + np.sqrt(DPY) * np.cumsum(eps, axis=1) / np.arange(1, N + 1)


def _alpha(h):
    return 1.0 - 2.0 ** (-1.0 / np.asarray(h, dtype=float))


def smooth_all(raw):
    # raw: (P, N) -> dict id -> (P, N). All filters in one time loop.
    P = raw.shape[0]
    kinds = SERIES[1:]
    F = len(kinds)
    n = np.arange(1, N + 1)
    # per-filter alpha schedule, shape (F, N)
    A = np.empty((F, N))
    for f, (_, _, kind, par, _, _) in enumerate(kinds):
        a = _alpha(par) if kind == "ema" else _alpha(np.maximum(3.0, n / par))
        A[f] = np.maximum(a, 1.0 / n)          # running mean while 1/n > alpha
    Y = np.empty((F, P, N))
    acc = np.repeat(raw[None, :, 0], F, axis=0)  # (F, P)
    Y[:, :, 0] = acc
    for i in range(1, N):
        acc += A[:, i, None] * (raw[None, :, i] - acc)
        Y[:, :, i] = acc
    out = {"raw": raw}
    for f, s in enumerate(kinds):
        out[s[0]] = Y[f]
    return out

## Ensemble statistics

`ensemble` returns two things. The first is the summary table. The second is the running ratios to raw, accumulated from inception up to each observation date — these are what the lower plots show. Results are cached by $(S, k)$.

One property of the model to keep in mind: the error $y - S$ does not depend on $S$ at all, since both the estimator and every filter are affine with weights summing to one. So the RMSE columns stay the same as you move the slider. Only turnover in $T\hat S$ changes, through the deterministic drift of $T\,S$.

In [4]:
_ENS_CACHE = {}

def ensemble(S, k, n_paths=400, seed=1000):
    key = (round(S, 6), k, n_paths, seed)
    if key in _ENS_CACHE:
        return _ENS_CACHE[key]
    d = smooth_all(raw_paths(S, n_paths, seed))
    grid = np.arange(0, N, k)
    Tg = T[grid]
    rows, run = [], {"rmse": [{}, {}], "turn": [{}, {}]}
    cum = {}
    for sid in IDS:
        y = d[sid][:, grid]
        e = y - S
        se, seT = (e ** 2).sum(0), ((Tg * e) ** 2).sum(0)                  # per grid point
        dS = np.abs(np.diff(y, axis=1)).sum(0)
        dTS = np.abs(np.diff(Tg * y, axis=1)).sum(0)
        cum[sid] = dict(se=np.cumsum(se), seT=np.cumsum(seT),
                        dS=np.concatenate([[0], np.cumsum(dS)]),
                        dTS=np.concatenate([[0], np.cumsum(dTS)]))
    G = len(grid)
    base = cum["raw"]
    with np.errstate(invalid="ignore", divide="ignore"):
        for sid in IDS:
            c = cum[sid]
            run["rmse"][0][sid] = np.sqrt(c["se"] / base["se"])
            run["rmse"][1][sid] = np.sqrt(c["seT"] / base["seT"])
            run["turn"][0][sid] = np.where(base["dS"] > 0, c["dS"] / base["dS"], 1.0)
            run["turn"][1][sid] = np.where(base["dTS"] > 0, c["dTS"] / base["dTS"], 1.0)
    for sid in IDS:
        c = cum[sid]
        v = c["se"][-1] / (n_paths * G)
        rows.append(dict(
            series=LABEL[sid],
            RMSE_S=np.sqrt(v),
            RMSE_TS=np.sqrt(c["seT"][-1] / (n_paths * G)),
            turnover_TS=c["dTS"][-1] / (n_paths * (G - 1)) * DPY / k,
            T_eff=1.0 / v,
        ))
    df = pd.DataFrame(rows).set_index("series")
    df.insert(1, "S_ratio", df["RMSE_S"] / df.loc["raw Ŝ", "RMSE_S"])
    df.insert(3, "TS_ratio", df["RMSE_TS"] / df.loc["raw Ŝ", "RMSE_TS"])
    df.insert(5, "x_less_turnover", df.loc["raw Ŝ", "turnover_TS"] / df["turnover_TS"])
    out = (df, run, grid)
    _ENS_CACHE[key] = out
    return out


def styled_table(df, k):
    lab = {1: "daily", 5: "every 5 d", 21: "every 21 d", 63: "every 63 d"}[k]
    t = df.rename(columns={
        "RMSE_S": "RMSE Ŝ", "S_ratio": "Ŝ ÷ raw", "RMSE_TS": "RMSE T·Ŝ",
        "TS_ratio": "T·Ŝ ÷ raw", "turnover_TS": f"turnover T·Ŝ ({lab})",
        "x_less_turnover": "× less turnover", "T_eff": "T_eff (y)"})

    def tint(v):
        if abs(v - 1) < 5e-4:
            return ""
        return "color:#cf5a22" if v > 1 else "color:#2a78d6"

    return (t.style
            .format({"RMSE Ŝ": "{:.4f}", "Ŝ ÷ raw": "{:.3f}", "RMSE T·Ŝ": "{:.3f}",
                     "T·Ŝ ÷ raw": "{:.3f}", f"turnover T·Ŝ ({lab})": "{:.2f}",
                     "× less turnover": "{:.1f}×", "T_eff (y)": "{:.2f}"})
            .map(tint, subset=["Ŝ ÷ raw", "T·Ŝ ÷ raw"]))

## Plots

In [5]:
LAYOUT = dict(template="plotly_white", height=640, hovermode="x unified",
              margin=dict(l=60, r=20, t=50, b=40),
              legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
              font=dict(family="Inter, Segoe UI, sans-serif", size=12))
YMIN_T = 0.25     # y-range ignores the first weeks, where 1/sqrt(T) is enormous


def _yrange(arrs, idx):
    m = T[idx] >= YMIN_T
    lo = min(np.nanmin(a[idx][m]) for a in arrs)
    hi = max(np.nanmax(a[idx][m]) for a in arrs)
    pad = 0.06 * (hi - lo)
    return [lo - pad, hi + pad]


def fig_paths(S, seed, k, visible, band=True):
    # One realization: S-hat on top, T*S-hat below.
    d = smooth_all(raw_paths(S, 1, seed))
    idx = np.arange(0, N, k)
    if idx[-1] != N - 1:
        idx = np.append(idx, N - 1)
    x = T[idx]
    se = 1 / np.sqrt(T[idx])
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=("Ŝ — annualized Sharpe estimate",
                                        "T·Ŝ — allocation score"))
    for row, scale in ((1, np.ones_like(x)), (2, x)):
        if band:
            fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]],
                                     y=np.r_[scale * (S + se), (scale * (S - se))[::-1]],
                                     fill="toself", fillcolor="rgba(21,23,27,0.06)",
                                     line=dict(width=0), hoverinfo="skip",
                                     name="±1 SE", showlegend=(row == 1)), row, 1)
        fig.add_trace(go.Scatter(x=x, y=scale * S, name=f"true S = {S:.2f}",
                                 line=dict(color="#15171b", width=1.3, dash="dash"),
                                 showlegend=(row == 1), legendgroup="truth",
                                 hovertemplate="%{y:.3f}"), row, 1)
        for sid in IDS:
            if sid not in visible:
                continue
            y = d[sid][0, idx] * scale
            fig.add_trace(go.Scatter(x=x, y=y, name=LABEL[sid], legendgroup=sid,
                                     showlegend=(row == 1),
                                     line=dict(color=COLOR[sid], dash=DASH[sid],
                                               width=1.1 if sid == "raw" else 1.8),
                                     hovertemplate="%{y:.3f}"), row, 1)
    vis = [d[s][0] for s in visible] or [d["raw"][0]]
    fig.update_yaxes(range=_yrange(vis + [S + 1 / np.sqrt(T), S - 1 / np.sqrt(T)], idx),
                     row=1, col=1, title_text="Ŝ")
    fig.update_yaxes(range=_yrange([T * v for v in vis] + [T * (S + 1 / np.sqrt(T)),
                                    T * (S - 1 / np.sqrt(T))], idx),
                     row=2, col=1, title_text="T·Ŝ")
    fig.update_xaxes(range=[0, YEARS], row=2, col=1, title_text="years since inception")
    fig.update_layout(**LAYOUT, title=f"One realization · seed {seed} · every {k} d")
    return fig


def fig_running(S, k, visible, metric="rmse", n_paths=400):
    # Ensemble running ratios to raw, in S-hat space (top) and T*S-hat space (bottom).
    _, run, grid = ensemble(S, k, n_paths)
    x = T[grid]
    name = "cumulative RMSE ÷ raw" if metric == "rmse" else "cumulative turnover ÷ raw"
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=("in Ŝ space", "in T·Ŝ space"))
    for row in (1, 2):
        R = run[metric][row - 1]
        fig.add_hline(y=1.0, line=dict(color="#8d928f", dash="dash", width=1.2), row=row, col=1)
        m = x >= YMIN_T
        vals = []
        for sid in IDS:
            if sid == "raw" or sid not in visible:
                continue
            vals.append(R[sid][m])
            fig.add_trace(go.Scatter(x=x, y=R[sid], name=LABEL[sid], legendgroup=sid,
                                     showlegend=(row == 1),
                                     line=dict(color=COLOR[sid], dash=DASH[sid], width=1.8),
                                     hovertemplate="%{y:.3f}"), row, 1)
        if vals:
            lo = min(1.0, min(np.nanmin(v) for v in vals))
            hi = max(1.0, max(np.nanmax(v) for v in vals))
            pad = 0.08 * (hi - lo) or 0.05
            fig.update_yaxes(range=[lo - pad, hi + pad], row=row, col=1, title_text="ratio to raw")
    fig.update_xaxes(range=[0, YEARS], row=2, col=1, title_text="years since inception")
    fig.update_layout(**LAYOUT, title=f"{name} · ensemble of {n_paths} paths · every {k} d")
    return fig

## Interactive panel

Controls: true Sharpe, realization seed (with a button to draw a new path), observation frequency, which series to show, whether to show the SE band, and the metric for the running plots. The top figure is one realization. The table and the lower figure use the 400-path ensemble.

The panel uses `ipywidgets` together with `fig.show()` inside an `Output` area. This works in JupyterLab, classic Notebook, VS Code and Colab, and doesn't need `FigureWidget` or `anywidget`.

In [6]:
def build_panel():
    s_true  = W.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05, description="true S",
                            continuous_update=False, readout_format=".2f")
    seed    = W.IntText(value=7, description="seed", layout=W.Layout(width="160px"))
    reseed  = W.Button(description="Draw a new path", icon="refresh")
    freq    = W.ToggleButtons(options=list(FREQS), value="3m", description="observe",
                              style={"button_width": "52px"})
    series  = W.SelectMultiple(options=[(LABEL[i], i) for i in IDS], value=tuple(IDS),
                               rows=len(IDS), description="series")
    band    = W.Checkbox(value=True, description="±1 SE band")
    metric  = W.ToggleButtons(options=[("RMSE", "rmse"), ("turnover", "turn")], value="rmse",
                              description="running", style={"button_width": "80px"})
    out_paths, out_table, out_run = W.Output(), W.Output(), W.Output()

    def render(*_):
        k = FREQS[freq.value]
        vis = list(series.value)
        with out_paths:
            clear_output(wait=True)
            fig_paths(s_true.value, seed.value, k, vis, band.value).show()
        df, _, _ = ensemble(s_true.value, k)
        with out_table:
            clear_output(wait=True)
            display(styled_table(df, k))
        with out_run:
            clear_output(wait=True)
            fig_running(s_true.value, k, vis, metric.value).show()

    def on_reseed(_):
        seed.value = int(np.random.default_rng().integers(1, 10**6))

    reseed.on_click(on_reseed)
    for w in (s_true, seed, freq, series, band, metric):
        w.observe(render, names="value")

    controls = W.HBox([W.VBox([s_true, W.HBox([seed, reseed]), freq, band, metric]), series])
    display(W.VBox([controls, out_paths, out_table, out_run]))
    render()


if HAVE_WIDGETS:
    build_panel()

## Static views

These cells run without widgets, which is useful for exporting or sharing a rendered notebook. Change the arguments to explore.

In [7]:
fig_paths(S=0.4, seed=7, k=63, visible=IDS).show()

In [8]:
df, _, _ = ensemble(S=0.4, k=1)
styled_table(df, 1)

,RMSE Ŝ,Ŝ ÷ raw,RMSE T·Ŝ,T·Ŝ ÷ raw,turnover T·Ŝ (daily),× less turnover,T_eff (y)
series,,,,,,,
raw Ŝ,0.9135,1.000,2.120,1.000,12.67,1.0×,1.20
EMA 1w,0.9876,1.081,2.122,1.001,3.38,3.7×,1.03
EMA 1m,1.0536,1.153,2.133,1.006,1.73,7.3×,0.90
EMA 3m,1.1025,1.207,2.165,1.021,1.10,11.5×,0.82
EMA 6m,1.1312,1.238,2.223,1.049,0.87,14.5×,0.78
EMA 1y,1.1591,1.269,2.367,1.117,0.74,17.2×,0.74
EMA h=N/60,0.9674,1.059,2.130,1.005,2.05,6.2×,1.07
EMA h=N/30,0.9693,1.061,2.141,1.010,1.54,8.2×,1.06
EMA h=N/15,0.9740,1.066,2.167,1.022,1.16,10.9×,1.05


In [9]:
fig_running(S=0.4, k=1, visible=IDS, metric="rmse").show()

### How the turnover advantage depends on observation frequency

RMSE barely changes with the observation frequency. Turnover changes a lot. Sampling every $k$ days is already a low-pass filter: raw turnover falls roughly as $1/\sqrt{k}$, while a filter whose halflife is shorter than $k$ has little left to remove.

In [10]:
cut = pd.DataFrame({lab: ensemble(0.4, k)[0]["x_less_turnover"] for lab, k in FREQS.items()})
cut.drop(index="raw Ŝ").style.format("{:.1f}×").set_caption("× less turnover vs raw, by observation frequency (S = 0.4)")

,1d,1w,1m,3m
series,,,,
EMA 1w,3.7×,1.9×,1.2×,1.0×
EMA 1m,7.3×,3.4×,1.8×,1.2×
EMA 3m,11.5×,5.2×,2.6×,1.6×
EMA 6m,14.5×,6.5×,3.3×,2.0×
EMA 1y,17.2×,7.7×,3.8×,2.3×
EMA h=N/60,6.2×,2.9×,1.7×,1.2×
EMA h=N/30,8.2×,3.8×,2.1×,1.5×
EMA h=N/15,10.9×,5.0×,2.7×,1.7×


### How the adaptive advantage depends on where you start measuring

The table above pools every date from inception. If you evaluate only from year 2, the fixed halflives look much better, because almost all of their cost is paid in the first months.

In [11]:
def rmse_ratio_from(S, start_year, n_paths=400, seed=1000):
    d = smooth_all(raw_paths(S, n_paths, seed))
    i0 = int(start_year * DPY)
    r0 = np.sqrt(((d["raw"][:, i0:] - S) ** 2).mean())
    return pd.Series({LABEL[s]: np.sqrt(((d[s][:, i0:] - S) ** 2).mean()) / r0 for s in IDS})

pd.DataFrame({f"from year {y}": rmse_ratio_from(0.4, y) for y in (0, 0.5, 2)}) \
  .style.format("{:.3f}").set_caption("RMSE Ŝ ÷ raw, by evaluation start (daily)")

,from year 0,from year 0.5,from year 2
raw Ŝ,1.000,1.000,1.000
EMA 1w,1.081,1.004,1.002
EMA 1m,1.153,1.020,1.008
EMA 3m,1.207,1.100,1.029
EMA 6m,1.238,1.195,1.070
EMA 1y,1.269,1.284,1.188
EMA h=N/60,1.059,1.006,1.006
EMA h=N/30,1.061,1.012,1.013
EMA h=N/15,1.066,1.025,1.026
